# Beyond Audio — Reproducible Final Experiments: Stage 3
## Visual typicality, error transitions, and artist-cluster bootstrap

**Project:** *Beyond Audio: Music Genre Classification*  
**Purpose:** Reconstruct the descriptive **visual-typicality analysis** and artist-cluster bootstrap uncertainty estimates reported in the paper.

**Scientific provenance.** This notebook uses the **original saved test-prediction CSV files** for CNN and MM classifiers, rather than silently substituting Stage 1 retrained predictions. Stage 1 found one differing test prediction for CNN early fusion, so using that retrained model would change a small number of retrospective statistics. The saved checkpoint predictions were independently verified against their original CSVs. The visual-typicality calculation, transitions, and bootstrap statistics are recomputed below and compared with the original artifacts when available.

**Important methodological limitations:**
- Each test item is assigned a typicality quintile **within its true genre** from cosine distance to its *training-set genre centroid* in **CNN visual embedding space**, after fitting `StandardScaler` on the training data only.
- Q1 is most visually typical; Q5 least. Both CNN and MM analyses use the **same** CNN-derived quintiles.
- True test labels are used **only for retrospective analysis**, not to make predictions or select classifiers. This is **not a deployable confidence score** and does not establish that visual typicality causes classification errors.
- Bootstrap samples whole artists with replacement (3,000 replicates, seed 2026) and uses **paired** differences for comparing models and for Q1 minus Q5.
- Outputs go to `finetuning_experiments/reproducible_stage3_2026/`. **Original inputs and outputs are never overwritten.**
- This notebook uses already evaluated test predictions to **verify previously reported descriptive analyses**. It should not be used to choose new model parameters after inspecting test outcomes.

**Recommended execution:** Run sections 1–3 first and inspect their outputs; then continue to the transition and bootstrap sections after validating quintile assignments.

## 1. Connect Google Drive and locate the original inputs

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import paired_cosine_distances

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = Path(os.environ.get("MSDI_PROJECT_DIR", "/content/drive/MyDrive/MSD-I_project"))
CNN_DIR = PROJECT / "finetuning_experiments" / "multimodal_artist_disjoint"
MM_DIR = PROJECT / "finetuning_experiments" / "oramas_representation_experiment"
PHASE2_DIR = PROJECT / "finetuning_experiments" / "phase2_error_analysis"
PHASE3_DIR = PROJECT / "finetuning_experiments" / "phase3_uncertainty"
OUTPUT = PROJECT / "finetuning_experiments" / "reproducible_stage3_2026"

SIZES = {"train": 8242, "test": 1792}

required_files = [
    CNN_DIR / "labels_train.csv",
    CNN_DIR / "labels_test.csv",
    MM_DIR / "labels_test.csv",
    CNN_DIR / "X_train_visual.npy",
    CNN_DIR / "X_test_visual.npy",
    CNN_DIR / "final_test_predictions.csv",
    MM_DIR / "representation_final_test_predictions.csv",
]
missing = [p for p in required_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing original input files:\n" + "\n".join(map(str, missing)))

print("Project root:", PROJECT)
print("CNN inputs:", CNN_DIR)
print("MM inputs:", MM_DIR)
print("New output location (not created yet):", OUTPUT)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)
print("All Stage 3 input files found.")

Mounted at /content/drive
Project root: /content/drive/MyDrive/MSD-I_project
CNN inputs: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/multimodal_artist_disjoint
MM inputs: /content/drive/MyDrive/MSD-I_project/finetuning_experiments/oramas_representation_experiment
New output location (not created yet): /content/drive/MyDrive/MSD-I_project/finetuning_experiments/reproducible_stage3_2026
Python: 3.13.18
NumPy: 2.4.6 pandas: 2.2.3 scikit-learn: 1.6.1
All Stage 3 input files found.


## 2. Verify artist-disjoint metadata and original prediction files

In [2]:
train_labels = pd.read_csv(CNN_DIR / "labels_train.csv", dtype={"msd_track_id": str, "msd_artist_id": str})
test_labels = pd.read_csv(CNN_DIR / "labels_test.csv", dtype={"msd_track_id": str, "msd_artist_id": str})
mm_test_labels = pd.read_csv(MM_DIR / "labels_test.csv", dtype={"msd_track_id": str, "msd_artist_id": str})
cnn_pred = pd.read_csv(CNN_DIR / "final_test_predictions.csv", dtype={"msd_track_id": str, "msd_artist_id": str})
mm_pred = pd.read_csv(MM_DIR / "representation_final_test_predictions.csv", dtype={"msd_track_id": str, "msd_artist_id": str})

meta_cols = ("msd_track_id", "msd_artist_id", "genre", "split")
for name, frame, expected_n, expected_split in [
    ("CNN train labels", train_labels, 8242, "train"),
    ("CNN test labels", test_labels, 1792, "test"),
    ("MM test labels", mm_test_labels, 1792, "test"),
]:
    assert set(meta_cols).issubset(frame.columns), f"Missing columns in {name}"
    assert len(frame) == expected_n, (name, len(frame))
    assert frame["msd_track_id"].is_unique, f"Duplicate tracks in {name}"
    assert frame[list(meta_cols)].notna().all().all(), name
    assert (frame["split"] == expected_split).all(), name

assert set(train_labels["msd_artist_id"]).isdisjoint(set(test_labels["msd_artist_id"])), "Artist overlap"
assert set(train_labels["msd_track_id"]).isdisjoint(set(test_labels["msd_track_id"])), "Track overlap"
for col in meta_cols:
    assert test_labels[col].equals(mm_test_labels[col]), f"CNN / MM metadata mismatch: {col}"

for name, pred, required_cols in [
    ("CNN predictions", cnn_pred, {"audio_prediction", "early_fusion_prediction"}),
    ("MM predictions", mm_pred, {"mm_audio_prediction", "mm_fusion_prediction"}),
]:
    assert "msd_track_id" in pred.columns and required_cols.issubset(pred.columns), name
    assert len(pred) == 1792 and pred["msd_track_id"].is_unique, name
    assert set(pred["msd_track_id"]) == set(test_labels["msd_track_id"]), name

# Match by unique ID, not implicit row position.
source = test_labels[list(meta_cols)].copy()
for name, pred, fields, prefix in [
    ("CNN", cnn_pred, ["audio_prediction", "early_fusion_prediction"], "cnn"),
    ("MM", mm_pred, ["mm_audio_prediction", "mm_fusion_prediction"], "mm"),
]:
    subset = pred[["msd_track_id"] + fields].copy()
    subset = subset.rename(columns={c: f"source_{prefix}_{c}" for c in fields})
    source = source.merge(subset, on="msd_track_id", how="left", validate="one_to_one", sort=False)

assert len(source) == 1792
assert source.notna().all().all(), "Missing predictions after ID alignment"
classes = sorted(train_labels["genre"].unique())
assert classes == sorted(test_labels["genre"].unique())
for col in source.columns:
    if col.startswith("source_"):
        assert set(source[col]).issubset(set(classes)), (col, set(source[col]) - set(classes))

print("Verified test tracks:", len(source))
print("Unique test artists:", source["msd_artist_id"].nunique())
print("Original saved predictions aligned by msd_track_id: VERIFIED")
print("Train/test artist disjointness: VERIFIED")
print("Genres:", classes)

Verified test tracks: 1792
Unique test artists: 975
Original saved predictions aligned by msd_track_id: VERIFIED
Train/test artist disjointness: VERIFIED
Genres: ['Electronic', 'Jazz', 'Metal', 'Pop', 'Rap', 'Rock']


## 3. Reconstruct genre-relative visual typicality (without reading saved quintiles)

**Definition:** Fit a scaler on CNN **training visual embeddings** only; compute each training genre's centroid in standardized visual space; calculate test-track cosine distance to the centroid of its **true** genre; rank the distances **within each genre** and partition into five almost equal groups.

The original `final_test_typicality_details.csv` is loaded **only at the end of this section**, to compare the independently reconstructed assignments with the existing report artifacts.

In [3]:
X_train_visual = np.load(CNN_DIR / "X_train_visual.npy", allow_pickle=False)
X_test_visual = np.load(CNN_DIR / "X_test_visual.npy", allow_pickle=False)
assert X_train_visual.shape == (8242, 2048)
assert X_test_visual.shape == (1792, 2048)
assert np.isfinite(X_train_visual).all() and np.isfinite(X_test_visual).all()

visual_scaler = StandardScaler()
Z_train = visual_scaler.fit_transform(X_train_visual)
Z_test = visual_scaler.transform(X_test_visual)

centroids = {
    genre: Z_train[train_labels["genre"].to_numpy() == genre].mean(axis=0)
    for genre in classes
}
matched_centroids = np.stack([centroids[genre] for genre in test_labels["genre"]])
distances = paired_cosine_distances(Z_test, matched_centroids)
assert np.isfinite(distances).all()

details = test_labels[list(meta_cols)].copy()
details["visual_distance"] = distances
details["typicality_quintile"] = (
    details.groupby("genre")["visual_distance"]
    .transform(lambda x: pd.qcut(x.rank(method="first"), q=5, labels=False) + 1)
    .astype(int)
)
assert set(details["typicality_quintile"]) == {1, 2, 3, 4, 5}

# Preserve row ordering and verify the original CSV only AFTER reconstructing.
original_typicality_path = CNN_DIR / "final_test_typicality_details.csv"
if original_typicality_path.is_file():
    original_typicality = pd.read_csv(original_typicality_path, dtype={"msd_track_id": str})
    original_typicality = original_typicality[["msd_track_id", "visual_distance", "typicality_quintile"]]
    assert original_typicality["msd_track_id"].is_unique
    aligned = details.merge(
        original_typicality, on="msd_track_id", how="left",
        suffixes=("_new", "_original"), validate="one_to_one", sort=False
    )
    assert len(aligned) == len(details)
    assert aligned[["visual_distance_original", "typicality_quintile_original"]].notna().all().all()
    n_different = int((aligned["typicality_quintile_new"] != aligned["typicality_quintile_original"]).sum())
    max_dist_difference = float(np.abs(aligned["visual_distance_new"] - aligned["visual_distance_original"]).max())
    print("Different quintile assignments vs original:", n_different)
    print("Maximum cosine-distance difference vs original:", f"{max_dist_difference:.12g}")
    if n_different:
        print(aligned.loc[aligned["typicality_quintile_new"] != aligned["typicality_quintile_original"],
            ["msd_track_id", "genre", "typicality_quintile_new", "typicality_quintile_original"]].head(15).to_string(index=False))
    assert n_different == 0, "Quintile mismatch: stop and investigate before subsequent sections"
else:
    print("Original typicality CSV not found; independent quintiles reconstructed but not cross-checked.")

print("\nVisual-typicality quintile sizes (Q1 most typical, Q5 least):")
print(details["typicality_quintile"].value_counts().sort_index().to_string())
print("\nGenre-by-quintile counts:")
print(pd.crosstab(details["genre"], details["typicality_quintile"]).to_string())
print("Note: true test genres were used only for retrospective typicality analysis.")

Different quintile assignments vs original: 0
Maximum cosine-distance difference vs original: 5.92803954813e-08

Visual-typicality quintile sizes (Q1 most typical, Q5 least):
typicality_quintile
1    361
2    357
3    357
4    357
5    360

Genre-by-quintile counts:
typicality_quintile    1    2    3    4    5
genre                                       
Electronic            76   75   76   75   76
Jazz                  46   46   45   46   46
Metal                 31   30   30   30   30
Pop                   52   52   51   52   52
Rap                   35   34   35   34   35
Rock                 121  120  120  120  121
Note: true test genres were used only for retrospective typicality analysis.


## 4. Derive paired rescued/harmed transitions from original saved predictions

**Rescued:** audio-only wrong, early fusion correct.  
**Harmed:** audio-only correct, early fusion wrong.  
The net accuracy change in percentage points is `100 × (rescued − harmed) / N`. This identity is checked both overall and by quintile.

In [4]:
# Join independently reconstructed quintiles and original saved predictions by track ID.
paired = details.merge(
    source.drop(columns=["msd_artist_id", "genre", "split"]),
    on="msd_track_id", how="left", validate="one_to_one", sort=False
)
assert len(paired) == 1792 and paired.notna().all().all()

prediction_columns = {
    "cnn_audio": "source_cnn_audio_prediction",
    "cnn_early_fusion": "source_cnn_early_fusion_prediction",
    "mm_audio": "source_mm_mm_audio_prediction",
    "mm_fusion": "source_mm_mm_fusion_prediction"
}
for model, pred_col in prediction_columns.items():
    paired[f"{model}_correct"] = (paired[pred_col].to_numpy() == paired["genre"].to_numpy())

transition_order = ["Both correct", "Rescued", "Harmed", "Both wrong"]
def classify_transitions(audio_correct, fusion_correct):
    a = np.asarray(audio_correct, dtype=bool)
    f = np.asarray(fusion_correct, dtype=bool)
    return np.select(
        [a & f, (~a) & f, a & (~f), (~a) & (~f)],
        transition_order, default="Unknown"
    )

paired["cnn_transition"] = classify_transitions(paired["cnn_audio_correct"], paired["cnn_early_fusion_correct"])
paired["mm_transition"] = classify_transitions(paired["mm_audio_correct"], paired["mm_fusion_correct"])
assert not (paired[["cnn_transition", "mm_transition"]] == "Unknown").any().any()

all_rows, quintile_rows = [], []
for rep, trans_col, audio_col, fusion_col in [
    ("CNN", "cnn_transition", "cnn_audio_correct", "cnn_early_fusion_correct"),
    ("MM", "mm_transition", "mm_audio_correct", "mm_fusion_correct")
]:
    counts = paired[trans_col].value_counts().reindex(transition_order, fill_value=0)
    rescued, harmed = int(counts["Rescued"]), int(counts["Harmed"])
    direct_gain = 100 * (paired[fusion_col].mean() - paired[audio_col].mean())
    transition_gain = 100 * (rescued - harmed) / len(paired)
    assert np.isclose(direct_gain, transition_gain)
    all_rows.append(dict(representation=rep, N=len(paired), both_correct=int(counts["Both correct"]),
                         rescued=rescued, harmed=harmed, both_wrong=int(counts["Both wrong"]),
                         net_gain_pp=transition_gain))
    for q, group in paired.groupby("typicality_quintile", sort=True):
        c = group[trans_col].value_counts().reindex(transition_order, fill_value=0)
        gain = 100 * (int(c["Rescued"]) - int(c["Harmed"])) / len(group)
        assert np.isclose(gain, 100 * (group[fusion_col].mean() - group[audio_col].mean()))
        quintile_rows.append(dict(representation=rep, quintile=int(q), N=len(group),
                                 both_correct=int(c["Both correct"]), rescued=int(c["Rescued"]),
                                 harmed=int(c["Harmed"]), both_wrong=int(c["Both wrong"]),
                                 rescue_pct=100 * int(c["Rescued"]) / len(group),
                                 harm_pct=100 * int(c["Harmed"]) / len(group), net_gain_pp=gain))

transition_overall = pd.DataFrame(all_rows)
transition_quintiles = pd.DataFrame(quintile_rows)
print("Overall audio → fusion transitions:")
print(transition_overall.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print("\nTransitions by visual-typicality quintile:")
print(transition_quintiles[["representation", "quintile", "N", "rescued", "harmed", "net_gain_pp"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

# Descriptive comparison with values stated in the current paper; never modify the computations to force a match.
reported_counts = {"CNN": (209, 142), "MM": (190, 124)}
for row in all_rows:
    expected = reported_counts[row["representation"]]
    print(f'{row["representation"]}: reconstructed rescued/harmed = {row["rescued"]}/{row["harmed"]};'
          f' paper values = {expected[0]}/{expected[1]}')

OUTPUT.mkdir(parents=True, exist_ok=True)
paired.to_csv(OUTPUT / "stage3_paired_test_predictions_and_typicality.csv", index=False)
transition_overall.to_csv(OUTPUT / "stage3_overall_transitions.csv", index=False)
transition_quintiles.to_csv(OUTPUT / "stage3_quintile_transitions.csv", index=False)
print("New analysis outputs saved only to:", OUTPUT)

Overall audio → fusion transitions:
representation    N  both_correct  rescued  harmed  both_wrong  net_gain_pp
           CNN 1792          1039      209     142         402         3.74
            MM 1792          1068      190     124         410         3.68

Transitions by visual-typicality quintile:
representation  quintile   N  rescued  harmed  net_gain_pp
           CNN         1 361       71       8        17.45
           CNN         2 357       54      10        12.32
           CNN         3 357       49      11        10.64
           CNN         4 357       25      32        -1.96
           CNN         5 360       10      81       -19.72
            MM         1 361       67       4        17.45
            MM         2 357       54       8        12.89
            MM         3 357       42      12         8.40
            MM         4 357       19      31        -3.36
            MM         5 360        8      69       -16.94
CNN: reconstructed rescued/harmed = 209/142

## 5. Artist-cluster bootstrap: overall and quintile fusion gains

This section reconstructs the original project’s **3,000-replicate** percentile bootstrap (`rng = default_rng(2026)`). Whole test **artists**, rather than individual tracks, are resampled with replacement. For each resample, the same sampled artists are used for both CNN and MM comparisons, preserving pairing. The original notebook generated quintile bootstrap draws before generating a separate set of overall bootstrap draws; this notebook keeps that order for verification.

In [5]:
artists = paired["msd_artist_id"].astype(str).to_numpy()
unique_artists, artist_codes = np.unique(artists, return_inverse=True)
quintiles = paired["typicality_quintile"].to_numpy(dtype=int)
assert len(paired) == 1792 and set(quintiles) == {1, 2, 3, 4, 5}

cnn_gain = (paired["cnn_early_fusion_correct"].to_numpy(dtype=int)
            - paired["cnn_audio_correct"].to_numpy(dtype=int))
mm_gain = (paired["mm_fusion_correct"].to_numpy(dtype=int)
           - paired["mm_audio_correct"].to_numpy(dtype=int))
gain = np.column_stack((cnn_gain, mm_gain, mm_gain - cnn_gain))
n_artists = len(unique_artists)
comparison_names = ["CNN fusion vs CNN audio", "MM fusion vs MM audio", "MM gain minus CNN gain"]

# Preaggregate by artist and true-genre visual-typicality quintile.
counts = np.zeros((n_artists, 5), dtype=np.int32)
sums = np.zeros((n_artists, 5, 3), dtype=float)
for i in range(len(paired)):
    a, q = artist_codes[i], quintiles[i] - 1
    counts[a, q] += 1
    sums[a, q, :] += gain[i, :]

n_boot = 3000
rng = np.random.default_rng(2026)
boot = np.full((n_boot, 5, 3), np.nan, dtype=float)
for b in range(n_boot):
    sampled = rng.integers(0, n_artists, size=n_artists)
    weights = np.bincount(sampled, minlength=n_artists)
    sample_counts = weights @ counts
    sample_sums = np.einsum("a,aqm->qm", weights, sums)
    valid = sample_counts > 0
    boot[b, valid, :] = 100 * sample_sums[valid, :] / sample_counts[valid, None]

original_n = counts.sum(axis=0)
original_sum = sums.sum(axis=0)
original = 100 * original_sum / original_n[:, None]
q_rows = []
for q in range(5):
    for m, name in enumerate(comparison_names):
        values = boot[:, q, m]
        values = values[np.isfinite(values)]
        lo, hi = np.percentile(values, [2.5, 97.5])
        q_rows.append(dict(quintile=q + 1, comparison=name, N=int(original_n[q]),
                           estimate_pp=original[q, m], CI_lower_pp=lo, CI_upper_pp=hi,
                           valid_bootstraps=len(values)))
quintile_ci = pd.DataFrame(q_rows)

# Advance the same RNG with new draws for the overall bootstrap (as in the original notebook).
overall_original = 100 * gain.mean(axis=0)
overall_boot = np.zeros((n_boot, 3), dtype=float)
artist_total_counts = counts.sum(axis=1)
artist_total_sums = sums.sum(axis=1)
for b in range(n_boot):
    sampled = rng.integers(0, n_artists, size=n_artists)
    weights = np.bincount(sampled, minlength=n_artists)
    n = weights @ artist_total_counts
    total = np.einsum("a,am->m", weights, artist_total_sums)
    overall_boot[b, :] = 100 * total / n

overall_ci = pd.DataFrame([
    dict(comparison=name, estimate_pp=overall_original[m],
         CI_lower_pp=np.percentile(overall_boot[:, m], 2.5),
         CI_upper_pp=np.percentile(overall_boot[:, m], 97.5))
    for m, name in enumerate(comparison_names)
])

print("Unique test artists:", n_artists, "| Replicates:", n_boot)
print("\nOverall fusion gains and 95% artist-cluster percentile CIs (pp):")
print(overall_ci.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print("\nFusion gains by visual-typicality quintile (pp):")
print(quintile_ci.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

quintile_ci.to_csv(OUTPUT / "stage3_quintile_fusion_gain_cluster_bootstrap.csv", index=False)
overall_ci.to_csv(OUTPUT / "stage3_overall_fusion_gain_cluster_bootstrap.csv", index=False)
np.save(OUTPUT / "stage3_quintile_bootstrap_samples.npy", boot)
print("Saved Stage 3 bootstrap files only to:", OUTPUT)

Unique test artists: 975 | Replicates: 3000

Overall fusion gains and 95% artist-cluster percentile CIs (pp):
             comparison  estimate_pp  CI_lower_pp  CI_upper_pp
CNN fusion vs CNN audio         3.74         1.62         5.86
  MM fusion vs MM audio         3.68         1.67         5.72
 MM gain minus CNN gain        -0.06        -1.83         1.67

Fusion gains by visual-typicality quintile (pp):
 quintile              comparison   N  estimate_pp  CI_lower_pp  CI_upper_pp  valid_bootstraps
        1 CNN fusion vs CNN audio 361        17.45        13.01        22.12              3000
        1   MM fusion vs MM audio 361        17.45        13.11        21.88              3000
        1  MM gain minus CNN gain 361         0.00        -3.56         3.56              3000
        2 CNN fusion vs CNN audio 357        12.32         7.85        16.85              3000
        2   MM fusion vs MM audio 357        12.89         8.83        17.08              3000
        2  MM gain

## 6. Paired Q1−Q5 contrasts and comparison against saved research outputs

Each Q1−Q5 bootstrap contrast is computed **within the same bootstrap replicate**. The first two rows describe whether fusion gain differs between most-typical and least-typical quintiles; the third describes the CNN-versus-MM difference in that contrast. The latter may have a confidence interval spanning zero even when each representation’s own Q1−Q5 contrast is substantial.

In [6]:
contrast_rows = []
for m, name in enumerate(comparison_names):
    q1 = original[0, m]
    q5 = original[4, m]
    diffs = boot[:, 0, m] - boot[:, 4, m]
    diffs = diffs[np.isfinite(diffs)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    contrast_rows.append(dict(comparison=name, Q1_gain_pp=q1, Q5_gain_pp=q5,
                              Q1_minus_Q5_pp=q1-q5, CI_lower_pp=lo, CI_upper_pp=hi,
                              valid_bootstraps=len(diffs)))
q1_q5_contrasts = pd.DataFrame(contrast_rows)
print("Paired Q1 minus Q5 contrasts (pp):")
print(q1_q5_contrasts.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
q1_q5_contrasts.to_csv(OUTPUT / "stage3_q1_q5_fusion_gain_contrasts.csv", index=False)

# Compare numerical columns with original saved bootstrap CSVs, if available.
comparisons = [
    (PHASE3_DIR / "quintile_fusion_gain_cluster_bootstrap.csv", quintile_ci,
     ["quintile", "comparison"], ["estimate_pp", "CI_lower_pp", "CI_upper_pp"]),
    (PHASE3_DIR / "overall_fusion_gain_cluster_bootstrap.csv", overall_ci,
     ["comparison"], ["estimate_pp", "CI_lower_pp", "CI_upper_pp"]),
    (PHASE3_DIR / "q1_q5_fusion_gain_contrasts.csv", q1_q5_contrasts,
     ["comparison"], ["Q1_minus_Q5_pp", "CI_lower_pp", "CI_upper_pp"]),
]
for path, fresh, keys, numeric_cols in comparisons:
    if not path.is_file():
        print("Not available for cross-check:", path.name)
        continue
    old = pd.read_csv(path)
    assert set(keys + numeric_cols).issubset(old.columns), f"Missing fields: {path.name}"
    merged = fresh[keys + numeric_cols].merge(
        old[keys + numeric_cols], on=keys, how="outer", validate="one_to_one",
        indicator=True, suffixes=("_new", "_original")
    )
    assert len(merged) == len(fresh) == len(old) and (merged["_merge"] == "both").all(), path.name
    diffs = {col: float(np.max(np.abs(merged[f"{col}_new"] - merged[f"{col}_original"])))
             for col in numeric_cols}
    print("Comparison with original", path.name, "maximum absolute differences:", diffs)

print("\nStage 3 reconstruction complete. Interpret the results as retrospective and exploratory.")

Paired Q1 minus Q5 contrasts (pp):
             comparison  Q1_gain_pp  Q5_gain_pp  Q1_minus_Q5_pp  CI_lower_pp  CI_upper_pp  valid_bootstraps
CNN fusion vs CNN audio       17.45      -19.72           37.17        30.64        43.70              3000
  MM fusion vs MM audio       17.45      -16.94           34.40        28.22        40.94              3000
 MM gain minus CNN gain        0.00        2.78           -2.78        -8.09         2.89              3000
Comparison with original quintile_fusion_gain_cluster_bootstrap.csv maximum absolute differences: {'estimate_pp': 2.220446049250313e-16, 'CI_lower_pp': 1.7763568394002505e-15, 'CI_upper_pp': 1.7763568394002505e-15}
Comparison with original overall_fusion_gain_cluster_bootstrap.csv maximum absolute differences: {'estimate_pp': 8.881784197001252e-16, 'CI_lower_pp': 0.0, 'CI_upper_pp': 0.0}
Comparison with original q1_q5_fusion_gain_contrasts.csv maximum absolute differences: {'Q1_minus_Q5_pp': 0.0, 'CI_lower_pp': 0.0, 'CI_upper_p

## Verification checklist and next steps

1. Sections 1–2 should find all original source files, confirm 1,792 test tracks, and validate artist disjointness.
2. Section 3 should print **zero differing typicality-quintile assignments** relative to the original CSV. Stop if this fails.
3. Section 4 should reproduce the saved transition counts, approximately **CNN 209 rescued / 142 harmed** and **MM 190 rescued / 124 harmed**.
4. Sections 5–6 should reproduce the paper’s 3,000-draw artist-cluster bootstrap estimates and paired Q1−Q5 contrasts. Small numerical differences in bootstrap percentile intervals should be investigated rather than concealed.
5. Keep public GitHub free of private Google Drive paths with identifying information, large licensed data, checkpoints, and unreviewed test-track metadata. The notebook code references only a generic project folder path; generated analysis CSVs stay in Drive by default.

**Not covered yet:** Figure reproduction, near-duplicate image sensitivity, 2,000-replicate bootstrap for MM-versus-CNN macro-F1, or independent clean-environment installation. Those are separate verification steps.